# 🌳 Decision Tree

Decision tree adalah model yang mengambil keputusan lewat **serangkaian pertanyaan ya/tidak**,
persis seperti cara kita mengenali bunga: *"kalau panjang petalnya di bawah 2 cm, berarti Setosa."*

**Studi kasus:** mengenali **spesies bunga iris** dari ukuran kelopaknya.

| Fitur | Arti |
|---|---|
| `sepal_length` | panjang kelopak luar (cm) |
| `sepal_width` | lebar kelopak luar (cm) |
| `petal_length` | panjang kelopak dalam (cm) |
| `petal_width` | lebar kelopak dalam (cm) |

**Target:** `species` — Setosa, Versicolor, atau Virginica (150 bunga, 50 tiap spesies).

Alur notebook:

1. Memuat & melihat data
2. Menyiapkan fitur & target
3. Melatih model decision tree
4. Evaluasi dengan akurasi & confusion matrix
5. Memprediksi bunga baru

> 💡 **Jalankan setiap sel** dengan memilihnya dan tekan `Shift + Enter`.

---
## 0️⃣ Setup & Import

Model decision tree sudah tersedia di scikit-learn — kita tidak perlu menulis algoritmanya sendiri.

> 📦 Jika scikit-learn belum terinstall: `uv add scikit-learn` atau `pip install scikit-learn`

In [1]:
# ── Import pustaka ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.colors import to_rgb
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

%matplotlib inline

# ── Warna & gaya grafik ──
# Tiga warna berbeda untuk tiga spesies — dipakai konsisten di semua grafik.
WARNA_KELAS = {"Setosa": "#2a78d6", "Versicolor": "#eb6834", "Virginica": "#1baf7a"}
BIRU, ORANYE = "#2a78d6", "#eb6834"   # untuk grafik dua seri

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "text.color": "#0b0b0b", "xtick.color": "#898781", "ytick.color": "#898781",
    "grid.color": "#e1e0d9", "axes.grid": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 110,
})

import sklearn
print(f"scikit-learn version : {sklearn.__version__}")
print(f"NumPy version        : {np.__version__}")
print(f"Pandas version       : {pd.__version__}")

scikit-learn version : 1.9.1
NumPy version        : 2.5.3
Pandas version       : 3.0.6


---
## 1️⃣ Memuat Dataset

Dataset `iris.csv` adalah dataset klasik di dunia machine learning: 150 bunga iris yang sudah
diukur dan diberi label spesiesnya.

In [2]:
# ── Membaca CSV ──
# Nama kolom di CSV memakai titik (petal.length) — kita rapikan jadi garis bawah.
df = pd.read_csv("data/iris.csv")
df.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]

print(f"Shape (baris, kolom) : {df.shape}")
print(f"Nama kolom           : {list(df.columns)}")

Shape (baris, kolom) : (150, 5)
Nama kolom           : ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']


In [3]:
# ── 5 baris pertama ──
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa
3,4.6,3.1,1.5,0.2,Setosa
4,5.0,3.6,1.4,0.2,Setosa


In [4]:
# ── Statistik deskriptif ──
FITUR  = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
TARGET = "species"

df[FITUR].describe().round(2)

,sepal_length,sepal_width,petal_length,petal_width
count,150.00,150.00,150.00,150.00
mean,5.84,3.06,3.76,1.20
std,0.83,0.44,1.77,0.76
min,4.30,2.00,1.00,0.10
25%,5.10,2.80,1.60,0.30
50%,5.80,3.00,4.35,1.30
75%,6.40,3.30,5.10,1.80
max,7.90,4.40,6.90,2.50


In [5]:
# ── Cek kualitas data & distribusi target ──
print("Jumlah nilai kosong per kolom:")
print(df.isna().sum())
print(f"\nTotal data kosong: {df.isna().sum().sum()}")

print("\nJumlah bunga per spesies:")
print(df[TARGET].value_counts())

Jumlah nilai kosong per kolom:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
species         0
dtype: int64

Total data kosong: 0

Jumlah bunga per spesies:
species
Setosa        50
Versicolor    50
Virginica     50
Name: count, dtype: int64


In [6]:
# ── Bentuk X (fitur) dan y (target) ──
# Targetnya berupa teks ("Setosa"), dan scikit-learn bisa menanganinya langsung.
X = df[FITUR]
y = df[TARGET]

print(f"X shape : {X.shape}  → {X.shape[0]} bunga, {X.shape[1]} fitur")
print(f"y shape : {y.shape}")
print(f"Kelas   : {list(y.unique())}")

X shape : (150, 4)  → 150 bunga, 4 fitur
y shape : (150,)
Kelas   : ['Setosa', 'Versicolor', 'Virginica']


In [7]:
# ── Bagi data: 80% train, 20% test ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% untuk test
    random_state=42,    # seed agar hasil bisa direproduksi
    stratify=y,         # jaga komposisi 50/50/50 tetap sama di kedua bagian
)

print(f"Total data : {len(X)}")
print(f"Data train : {len(X_train)} ({len(X_train)/len(X):.0%})")
print(f"Data test  : {len(X_test)} ({len(X_test)/len(X):.0%})")
print(f"\nKomposisi data test:")
print(y_test.value_counts())

Total data : 150
Data train : 120 (80%)
Data test  : 30 (20%)

Komposisi data test:
species
Setosa        10
Virginica     10
Versicolor    10
Name: count, dtype: int64



## 3️⃣ Melatih Model


Melatihnya hanya **dua baris** — sama seperti regresi linear, bedanya kita pakai
`DecisionTreeClassifier` karena targetnya berupa **kelas**, bukan angka.

Kita batasi `max_depth=3` supaya pohonnya tetap **ringkas dan mudah dibaca**.

In [8]:
# ── Latih model decision tree ──
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

print("Model selesai dilatih ✓")
print(f"Kedalaman pohon  : {model.get_depth()}")
print(f"Jumlah daun      : {model.get_n_leaves()}")

Model selesai dilatih ✓
Kedalaman pohon  : 3
Jumlah daun      : 5


---
## 4️⃣ Evaluasi Model

**Akurasi** = proporsi prediksi yang benar. Selalu bandingkan dengan **baseline**: model yang
selalu menebak satu spesies saja. Karena ketiga spesies jumlahnya sama banyak, baseline di sini
hanya **33%** (kode di bawah mengambil Setosa karena ia yang pertama secara urutan abjad).

In [9]:
# ── Akurasi train vs test ──
pred_train = model.predict(X_train)
pred_test  = model.predict(X_test)

akurasi_train = accuracy_score(y_train, pred_train)
akurasi_test  = accuracy_score(y_test, pred_test)

# Baseline: selalu menebak spesies mayoritas
kelas_mayoritas = y_train.mode()[0]
baseline = accuracy_score(y_test, np.full_like(y_test, kelas_mayoritas))

print(f"{'':12s} {'Akurasi':>9s}")
print("-" * 22)
print(f"{'Train':12s} {akurasi_train:>9.1%}")
print(f"{'Test':12s} {akurasi_test:>9.1%}")
print(f"\n{'Baseline':12s} {baseline:>9.1%}  (selalu menebak '{kelas_mayoritas}')")
print(f"\n→ Model mengalahkan baseline sebesar {akurasi_test - baseline:+.1%}")

               Akurasi
----------------------
Train            98.3%
Test             96.7%

Baseline         33.3%  (selalu menebak 'Setosa')

→ Model mengalahkan baseline sebesar +63.3%


In [10]:
# ── Confusion matrix: di mana model salah? ──
cm = confusion_matrix(y_test, pred_test, labels=model.classes_)

pd.DataFrame(
    cm,
    index=[f"asli {k}" for k in model.classes_],
    columns=[f"prediksi {k}" for k in model.classes_],
)

,prediksi Setosa,prediksi Versicolor,prediksi Virginica
asli Setosa,10,0,0
asli Versicolor,0,9,1
asli Virginica,0,0,10


In [11]:
# ── Laporan lengkap per kelas ──
print(classification_report(y_test, pred_test))

              precision    recall  f1-score   support

      Setosa       1.00      1.00      1.00        10
  Versicolor       1.00      0.90      0.95        10
   Virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



In [12]:
# ── Data bunga baru (hasil pengukuran dengan penggaris) ──
bunga_baru = pd.DataFrame([
    {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2, "nama": "Bunga A"},
    {"sepal_length": 6.0, "sepal_width": 2.7, "petal_length": 4.5, "petal_width": 1.4, "nama": "Bunga B"},
    {"sepal_length": 6.7, "sepal_width": 3.0, "petal_length": 5.8, "petal_width": 2.1, "nama": "Bunga C"},
    {"sepal_length": 5.9, "sepal_width": 3.0, "petal_length": 4.8, "petal_width": 1.7, "nama": "Bunga D"},
])

prediksi = model.predict(bunga_baru[FITUR])
peluang  = model.predict_proba(bunga_baru[FITUR])

hasil = pd.DataFrame(peluang, columns=[f"peluang {k}" for k in model.classes_])
hasil.insert(0, "nama", bunga_baru["nama"])
hasil["prediksi"] = prediksi
hasil.round(3)

,nama,peluang Setosa,peluang Versicolor,peluang Virginica,prediksi
0,Bunga A,1.0,0.000,0.000,Setosa
1,Bunga B,0.0,1.000,0.000,Versicolor
2,Bunga C,0.0,0.000,1.000,Virginica
3,Bunga D,0.0,0.333,0.667,Virginica
